In [0]:
df_apolice   = spark.read.format("delta").table("silver.apolice")
df_carro     = spark.read.format("delta").table("silver.carro")
df_cliente   = spark.read.format("delta").table("silver.cliente")
df_endereco  = spark.read.format("delta").table("silver.endereco")
df_estado    = spark.read.format("delta").table("silver.estado")
df_marca     = spark.read.format("delta").table("silver.marca")
df_modelo    = spark.read.format("delta").table("silver.modelo")
df_municipio = spark.read.format("delta").table("silver.municipio")
df_regiao    = spark.read.format("delta").table("silver.regiao")
df_sinistro  = spark.read.format("delta").table("silver.sinistro")
df_telefone  = spark.read.format("delta").table("silver.telefone")
 
print("✅ DataFrames lidos do Silver!")


In [0]:
df_apolice.createOrReplaceTempView("apolice")
df_carro.createOrReplaceTempView("carro")
df_cliente.createOrReplaceTempView("cliente")
df_endereco.createOrReplaceTempView("endereco")
df_estado.createOrReplaceTempView("estado")
df_marca.createOrReplaceTempView("marca")
df_modelo.createOrReplaceTempView("modelo")
df_municipio.createOrReplaceTempView("municipio")
df_regiao.createOrReplaceTempView("regiao")
df_sinistro.createOrReplaceTempView("sinistro")
df_telefone.createOrReplaceTempView("telefone")
 
print("✅ Temp Views criadas!")


In [0]:
%sql
 DROP TABLE IF EXISTS gold.dim_carro;
 

 CREATE TABLE gold.dim_carro (
   SK_CARRO    BIGINT GENERATED BY DEFAULT AS IDENTITY,
   PLACA       VARCHAR(10),
   MARCA       VARCHAR(100),
   MODELO      VARCHAR(100),
   COR         VARCHAR(10),
   ANO         INT,
   CHASSI      VARCHAR(20)
 ) USING DELTA;



In [0]:
%sql
 WITH carros_relacional AS (
     SELECT c.PLACA,
            ma.NOME_MARCA,
            mo.NOME_MODELO,
            c.COR,
            c.ANO,
            c.CHASSI
       FROM carro c
            INNER JOIN modelo mo ON c.CODIGO_MODELO = mo.CODIGO_MODELO
            INNER JOIN marca ma  ON mo.CODIGO_MARCA  = ma.CODIGO_MARCA
 )
 MERGE INTO gold.dim_carro AS c
 USING carros_relacional AS cc
 ON c.PLACA = cc.PLACA

 WHEN MATCHED AND (
     c.MARCA  <> cc.NOME_MARCA  OR
     c.MODELO <> cc.NOME_MODELO OR
     c.COR    <> cc.COR         OR
     c.ANO    <> cc.ANO         OR
     c.CHASSI <> cc.CHASSI
 ) THEN
     UPDATE SET PLACA  = cc.PLACA,
                MARCA  = cc.NOME_MARCA,
                MODELO = cc.NOME_MODELO,
                COR    = cc.COR,
                ANO    = cc.ANO,
                CHASSI = cc.CHASSI

 WHEN NOT MATCHED THEN
     INSERT (PLACA, MARCA, MODELO, COR, ANO, CHASSI)
     VALUES (cc.PLACA, cc.NOME_MARCA, cc.NOME_MODELO, cc.COR, cc.ANO, cc.CHASSI);

In [0]:
from pyspark.sql.functions import expr
 
# Define intervalo de datas
data_inicial = "2023-01-01"
data_final   = "2026-12-31"
 
# Calcula número de dias
num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]
 
# Cria DataFrame de datas
df_calendario = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS Data")
 
# Extrai componentes da data
df_tempo = df_calendario.selectExpr(
    "Data",
    "year(Data)  AS Ano",
    "month(Data) AS Mes",
    """(CASE month(Data)
        WHEN 1  THEN 'JANEIRO'
        WHEN 2  THEN 'FEVEREIRO'
        WHEN 3  THEN 'MARCO'
        WHEN 4  THEN 'ABRIL'
        WHEN 5  THEN 'MAIO'
        WHEN 6  THEN 'JUNHO'
        WHEN 7  THEN 'JULHO'
        WHEN 8  THEN 'AGOSTO'
        WHEN 9  THEN 'SETEMBRO'
        WHEN 10 THEN 'OUTUBRO'
        WHEN 11 THEN 'NOVEMBRO'
        WHEN 12 THEN 'DEZEMBRO'
    END) AS NomeMes""",
    "day(Data) AS Dia",
    """(CASE dayofweek(Data)
        WHEN 1 THEN 'DOMINGO'
        WHEN 2 THEN 'SEGUNDA-FEIRA'
        WHEN 3 THEN 'TERCA-FEIRA'
        WHEN 4 THEN 'QUARTA-FEIRA'
        WHEN 5 THEN 'QUINTA-FEIRA'
        WHEN 6 THEN 'SEXTA-FEIRA'
        WHEN 7 THEN 'SABADO'
    END) AS NomeDiaSemana""",
    "dayofweek(Data) AS NumeroDiaSemana"
)

df_tempo.display()
 
# Salvar DIM_TEMPO no Gold
spark.sql("DROP TABLE IF EXISTS gold.dim_tempo")
df_tempo.write.mode('overwrite').saveAsTable("gold.dim_tempo", format="delta")
 
print("✅ DIM_TEMPO criada!")



In [0]:
%sql
DROP TABLE IF EXISTS gold.dim_cliente;

CREATE TABLE gold.dim_cliente (
   SK_CLIENTE        BIGINT GENERATED BY DEFAULT AS IDENTITY,
   CODIGO_CLIENTE    INT,
   NOME              VARCHAR(50),
   CPF               VARCHAR(14), -- <--- O AJUSTE FOI FEITO AQUI
   SEXO              CHAR(1),
   DATA_NASCIMENTO   DATE
) USING DELTA;

In [0]:
%sql
WITH cliente_relacional AS (
     SELECT CODIGO_CLIENTE,
            NOME,
            CPF,
            SEXO,
            DATA_NASCIMENTO
       FROM cliente
 )
 MERGE INTO gold.dim_cliente AS d
 USING cliente_relacional AS r
 ON r.CODIGO_CLIENTE = d.CODIGO_CLIENTE

 WHEN MATCHED AND (
     r.NOME            <> d.NOME            OR
     r.CPF             <> d.CPF             OR
     r.SEXO            <> d.SEXO            OR
     r.DATA_NASCIMENTO <> d.DATA_NASCIMENTO
 ) THEN
     UPDATE SET CODIGO_CLIENTE  = r.CODIGO_CLIENTE,
                NOME            = r.NOME,
                CPF             = r.CPF,
                SEXO            = r.SEXO,
                DATA_NASCIMENTO = r.DATA_NASCIMENTO

 WHEN NOT MATCHED THEN
     INSERT (CODIGO_CLIENTE, NOME, CPF, SEXO, DATA_NASCIMENTO)
     VALUES (r.CODIGO_CLIENTE, r.NOME, r.CPF, r.SEXO, r.DATA_NASCIMENTO);


In [0]:
%sql
 DROP TABLE IF EXISTS gold.dim_localidade;
 

 CREATE TABLE gold.dim_localidade (
    SK_LOCALIDADE    BIGINT GENERATED BY DEFAULT AS IDENTITY,
    CODIGO_MUNICIPIO INT,
    NOME_MUNICIPIO   VARCHAR(100),
    NOME_ESTADO      VARCHAR(100),
    NOME_REGIAO      VARCHAR(100)
 ) USING DELTA;


In [0]:
%sql
 WITH localidade_relacional AS (
     SELECT m.CODIGO_MUNICIPIO,
            m.NOME_MUNICIPIO,
            e.NOME_ESTADO,
            r.NOME_REGIAO
       FROM municipio m
            INNER JOIN estado  e ON m.CODIGO_ESTADO = e.CODIGO_ESTADO
            INNER JOIN regiao  r ON e.CODIGO_REGIAO = r.CODIGO_REGIAO
 )
 MERGE INTO gold.dim_localidade AS d
 USING localidade_relacional AS r
 ON r.CODIGO_MUNICIPIO = d.CODIGO_MUNICIPIO

 WHEN MATCHED AND (
     r.NOME_MUNICIPIO <> d.NOME_MUNICIPIO OR
     r.NOME_ESTADO    <> d.NOME_ESTADO    OR
     r.NOME_REGIAO    <> d.NOME_REGIAO
 ) THEN
     UPDATE SET CODIGO_MUNICIPIO = r.CODIGO_MUNICIPIO,
                NOME_MUNICIPIO   = r.NOME_MUNICIPIO,
                NOME_ESTADO      = r.NOME_ESTADO,
                NOME_REGIAO      = r.NOME_REGIAO

 WHEN NOT MATCHED THEN
     INSERT (CODIGO_MUNICIPIO, NOME_MUNICIPIO, NOME_ESTADO, NOME_REGIAO)
     VALUES (r.CODIGO_MUNICIPIO, r.NOME_MUNICIPIO, r.NOME_ESTADO, r.NOME_REGIAO);


In [0]:
%sql
 DROP TABLE IF EXISTS gold.fato_sinistro;
 

 CREATE TABLE gold.fato_sinistro (
    FK_TEMPO        DATE,
    FK_LOCALIDADE   INT,
    FK_CARRO        INT,
    FK_CLIENTE      INT,
    QTDE_SINISTRO   INT
 ) USING DELTA;


In [0]:
%sql
 WITH apolice_cliente AS (
     SELECT a.CODIGO_CLIENTE, a.PLACA
       FROM apolice a
            INNER JOIN cliente c ON a.CODIGO_CLIENTE = c.CODIGO_CLIENTE
 )
 INSERT INTO gold.fato_sinistro
 SELECT s.DATA_SINISTRO,
        dloc.SK_LOCALIDADE,
        dcar.SK_CARRO,
        dcli.SK_CLIENTE,
        COUNT(1) AS QTDE_SINISTRO
   FROM sinistro s
        INNER JOIN gold.dim_carro      dcar ON s.PLACA          = dcar.PLACA
        INNER JOIN apolice_cliente     ac   ON ac.PLACA         = s.PLACA
        INNER JOIN gold.dim_cliente    dcli ON ac.CODIGO_CLIENTE = dcli.CODIGO_CLIENTE
        INNER JOIN gold.dim_localidade dloc ON s.LOCAL_SINISTRO = dloc.CODIGO_MUNICIPIO
        INNER JOIN gold.dim_tempo      dtem ON s.DATA_SINISTRO   = dtem.Data
 GROUP BY s.DATA_SINISTRO,
          dloc.SK_LOCALIDADE,
          dcar.SK_CARRO,
          dcli.SK_CLIENTE;


In [0]:
%sql
 SHOW TABLES IN gold;
 
 
 SELECT * FROM gold.dim_carro LIMIT 10;
 
 
 SELECT * FROM gold.dim_cliente LIMIT 10;
 
 
 SELECT * FROM gold.dim_localidade LIMIT 10;
 
 
 SELECT * FROM gold.dim_tempo LIMIT 10;
 
 
 SELECT * FROM gold.fato_sinistro LIMIT 10;
 

In [0]:
print("✅ Notebook Gold executado com sucesso!")
print("Tabelas criadas no Gold:")
print("  - gold.dim_carro")
print("  - gold.dim_tempo")
print("  - gold.dim_cliente")
print("  - gold.dim_localidade")
print("  - gold.fato_sinistro")